# Supply Generator

The Supply Generator module of the robin simulator produces a randomized list
of services and a supply YAML file from two configuration files. This notebook
demonstrates the `SupplyGenerator` class and visualizes the result with a
Marey chart.


## 0. Load Libraries


In [ ]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path

from robin.plotter.entities import KernelPlotter
from robin.supply.generator.entities import SupplyGenerator
from robin.supply.entities import Supply

## 1. Generate Supply

Two configuration files are required:
- **Supply configuration**: operational times, RUs, lines, stations, etc.
- **Generator configuration**: dates, prices, departure-time distributions, etc.


In [ ]:
supply_config_path = Path('../configs/supply_generator/supply_data.yaml')
generator_config_path = Path('../configs/supply_generator/config.yaml')
generator_save_path = Path('../data/results/supply_dummy.yaml')

Path('../data/results').mkdir(parents=True, exist_ok=True)

seed = 50
generator = SupplyGenerator.from_yaml(
    path_config_supply=supply_config_path,
    path_config_generator=generator_config_path,
)

By calling `generate()` a list of services is produced and the supply YAML is
saved. Services can be generated with or without conflicts.


In [ ]:
generator.generate(
    n_services=25,
    output_path=generator_save_path,
    seed=seed,
    progress_bar=True,
    without_conflicts=False,
)

print(f'Number of service requests generated: {len(generator.services)}')

## 2. Load Supply

Check that the generated YAML can be loaded back by the robin Supply module.


In [ ]:
supply = Supply.from_yaml(path=str(generator_save_path))
print(f'Loaded {len(supply.services)} services')

## 3. Marey Chart

The Marey chart visualizes the services matching the provided date, one plot
per corridor branch.


In [ ]:
kernel_plotter = KernelPlotter(path_config_supply=str(generator_save_path))

date = datetime.datetime.strptime('2024-06-25', '%Y-%m-%d').date()
kernel_plotter.plot_marey_chart(
    date=date,
    save_path='../reports/figures/',
)